In [2]:
import pygame
import random
import sys
import psycopg2

conn = psycopg2.connect(
    dbname="lab10",
    user="postgres",
    password="#Kazakhstan2050",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

def get_or_create_user(username):
    cur.execute("SELECT id FROM users WHERE username = %s", (username,))
    user = cur.fetchone()
    if user:
        user_id = user[0]
        cur.execute("SELECT level, score FROM user_score WHERE user_id = %s", (user_id,))
        row = cur.fetchone()
        if row:
            return user_id, row[0], row[1]
        else:
            cur.execute("INSERT INTO user_score (user_id, level, score) VALUES (%s, %s, %s)", (user_id, 1, 0))
            conn.commit()
            return user_id, 1, 0
    else:
        cur.execute("INSERT INTO users (username) VALUES (%s) RETURNING id", (username,))
        user_id = cur.fetchone()[0]
        cur.execute("INSERT INTO user_score (user_id, level, score) VALUES (%s, %s, %s)", (user_id, 1, 0))
        conn.commit()
        return user_id, 1, 0

def save_progress(user_id, level, score):
    cur.execute("UPDATE user_score SET level = %s, score = %s WHERE user_id = %s", (level, score, user_id))
    conn.commit()

username = input("Enter your username: ")
user_id, level, score = get_or_create_user(username)

pygame.init()
WIDTH, HEIGHT = 600, 600
BLOCK_SIZE = 20
WHITE = (255, 255, 255)
GREEN = (0, 255, 0)
RED = (255, 0, 0)
GRAY = (100, 100, 100)
BLACK = (0, 0, 0)

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Snake Game with Levels")
clock = pygame.time.Clock()
font = pygame.font.SysFont("Arial", 25)

def draw_block(color, pos):
    pygame.draw.rect(screen, color, pygame.Rect(pos[0], pos[1], BLOCK_SIZE, BLOCK_SIZE))

def get_random_position():
    return (
        random.randint(0, (WIDTH - BLOCK_SIZE) // BLOCK_SIZE) * BLOCK_SIZE,
        random.randint(0, (HEIGHT - BLOCK_SIZE) // BLOCK_SIZE) * BLOCK_SIZE
    )

def draw_walls(level):
    walls = []
    if level >= 2:
        for i in range(5, 25):
            walls.append((i * BLOCK_SIZE, 10 * BLOCK_SIZE))
    if level >= 3:
        for i in range(10, 20):
            walls.append((15 * BLOCK_SIZE, i * BLOCK_SIZE))
    for wall in walls:
        draw_block(GRAY, wall)
    return walls

snake = [(100, 100), (80, 100), (60, 100)]
direction = (BLOCK_SIZE, 0)
apple = get_random_position()
paused = False

running = True
while running:
    screen.fill(BLACK)
    clock.tick(10 + level * 2)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            save_progress(user_id, level, score)
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_p:
                paused = not paused

    if not paused:
        keys = pygame.key.get_pressed()
        if keys[pygame.K_UP] and direction != (0, BLOCK_SIZE):
            direction = (0, -BLOCK_SIZE)
        elif keys[pygame.K_DOWN] and direction != (0, -BLOCK_SIZE):
            direction = (0, BLOCK_SIZE)
        elif keys[pygame.K_LEFT] and direction != (BLOCK_SIZE, 0):
            direction = (-BLOCK_SIZE, 0)
        elif keys[pygame.K_RIGHT] and direction != (-BLOCK_SIZE, 0):
            direction = (BLOCK_SIZE, 0)

        head = (snake[0][0] + direction[0], snake[0][1] + direction[1])
        snake.insert(0, head)

        if head == apple:
            score += 1
            apple = get_random_position()
            level = min(score // 5 + 1, 3)
        else:
            snake.pop()

        draw_block(RED, apple)
        for segment in snake:
            draw_block(GREEN, segment)
        walls = draw_walls(level)

        if (
            head in snake[1:] or
            head[0] < 0 or head[0] >= WIDTH or
            head[1] < 0 or head[1] >= HEIGHT or
            head in walls
        ):
            text = font.render("Game Over!", True, WHITE)
            screen.blit(text, (WIDTH // 2 - 60, HEIGHT // 2))
            pygame.display.flip()
            pygame.time.wait(2000)
            save_progress(user_id, level, score)
            pygame.quit()
            sys.exit()

        score_text = font.render(f"Score: {score}  Level: {level}", True, WHITE)
        screen.blit(score_text, (10, 10))
    else:
        pause_text = font.render("Paused", True, WHITE)
        screen.blit(pause_text, (WIDTH // 2 - 40, HEIGHT // 2 - 20))

    pygame.display.flip()


pygame 2.6.1 (SDL 2.28.4, Python 3.11.4)
Hello from the pygame community. https://www.pygame.org/contribute.html
Enter your username: ttt


SystemExit: 

C:\Users\Nurdaulet\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
